# DPO: from the paper to `losses.py`

This notebook walks from the DPO objective (Rafailov et al., 2023, Eq. 7) to the ~20-line implementation in `losses.py`. The point is to make the math visible: what the loss does, what role the weighting term plays, and why the ablations behave the way they do.

## 1. The objective

For a prompt $x$ and a preference pair $(y_w, y_l)$ (chosen, rejected), DPO's loss is:

$$
\mathcal{L}_\text{DPO} = -\mathbb{E}\left[\log\sigma\bigl(\beta \,(r_\theta(x,y_w) - r_\theta(x,y_l))\bigr)\right]
$$

where the *implicit reward* is

$$
r_\theta(x,y) = \log\frac{\pi_\theta(y\mid x)}{\pi_\text{ref}(y\mid x)}.
$$

That's the entire algorithm. No reward model, no PPO, no KL penalty baked in as a separate term — the reference ratio *is* the KL-regularized optimal reward.

## 2. Why the sigmoid matters

A naive reading of DPO is: increase $\log \pi_\theta(y_w\mid x)$ and decrease $\log\pi_\theta(y_l\mid x)$. If we drop the sigmoid we get exactly that: the loss becomes $-\beta(\ldots)$, a linear margin.

The sigmoid down-weights pairs the policy already ranks correctly. Without it, well-separated pairs keep pulling on gradients and the ratios run away. See the `abl_dpo_no_weight` ablation — training degenerates.

In [ ]:
import math, jax.numpy as jnp
from losses import dpo_loss

# Sanity: policy == ref => loss = log 2 ≈ 0.6931
loss, m = dpo_loss(jnp.zeros(1), jnp.zeros(1), jnp.zeros(1), jnp.zeros(1), beta=0.1)
print(f'untrained loss: {float(loss):.4f}  (expected {math.log(2):.4f})')

## 3. Evaluate a trained checkpoint

Point `CKPT` at a step directory produced by `train.py`.

In [ ]:
CKPT = 'outputs/gpt2_cpu_smoke/step_000200'

import pickle, jax
from pathlib import Path
from datasets import load_dataset
from model import load_model_and_tokenizer
from eval import pairwise_accuracy

state = pickle.load(open(Path(CKPT) / 'state.pkl', 'rb'))
cfg = state['config']
lora_params = jax.tree_util.tree_map(jnp.asarray, state['lora_params'])

model, base_params, tokenizer = load_model_and_tokenizer(cfg['model'])
ds = load_dataset(cfg['dataset'], split='train[-64:]')
acc = pairwise_accuracy(model, base_params, lora_params, tokenizer, ds, cfg)
print(f'pairwise_accuracy on 64 held-out pairs: {acc:.3f}')

## 4. Ablation reading guide

Each ablation config in `configs/abl_*.yaml` isolates one assumption from the paper:

- `abl_dpo_no_ref` — drop the reference, keep the sigmoid. The policy can drift arbitrarily from the base distribution (KL explodes).
- `abl_dpo_no_weight` — keep the reference, drop the sigmoid. Linear-margin maximization; the paper's warned-about degenerate case.
- `abl_beta_bad` — $\beta=10$. Over-regularizes, loss stays near `log 2`, pairwise accuracy fails to rise.
- `abl_chosen_only` — SFT on chosen completions; no separation signal. Pairwise accuracy stays near chance.

If your implementation is correct, all four of these should behave worse than the main DPO config on held-out pairwise accuracy.